# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [21]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.7 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

1. Lecture 01 (Command line & Python basics): using python basics and saving the result directly into output/file_inventory.txt.

Lecture 02 (Git, functions, and files): Python functions like profile_table() 

Lecture 03 (Environments, shell pipelines, & NumPy): Computes per-visit systolic blood pressure averages using np.array, where axis=0 calculates the mean vertically down the rows for each column independently across all patients.

Lecture 04 (Jupyter & labeled pandas data): Learned using pandas DataFrame indexing and labeled operations to profile tables 

Lecture 05 (Data contracts, cleaning, & validation): Applies strict data validation rules, sentinel value handling (such as 9999 and 999)

Lecture 06 (Joins, reshaping, & concatenation): Combines row-wise table stacking (pd.concat) for visit files 

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [22]:
%%bash
wc -l data/*.csv > output/file_inventory.txt
cat output/file_inventory.txt

       6 data/sites.csv
   26298 data/visits_north.csv
   31075 data/visits_south.csv
   57379 total


### 1.2 Profile each table with a function

In [23]:
import pandas as pd
from pathlib import Path

def profile_table(name, df):
    return {
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicate_rows": int(df.duplicated().sum())
    }

results = []
for name in manifest["files"]:
    df = raw[Path(name).stem]
    results.append(profile_table(name, df))

profile_df = pd.DataFrame(results)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
profile_df.to_csv(PROFILE_PATH, index=False)
profile_df

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

- Patient: One row per unique registered patient. Primary key: `patient_id`. Raw violations include exact duplicate rows and invalid values.
- Sites: One row per current site record. Primary key: `site_code`. Raw violations include duplicate site code entries.
- Visits: One row per patient per visit (`baseline` or `week12`) after stacking the north and south office files and resolving clinic rechecks by retaining the later reading. Composite primary key: `(patient_id, visit)`. Raw violations include duplicate rows, unparsed dates, and rechecks on the same visit.
- A1c: One row per patient per visit in long format. Composite primary key: `(patient_id, visit)`.  Raw violations include duplicate site code entries.
- Join cardinalities:
  - `visits` to `patients`: Many-to-one
  - `visits` to `sites`: Many-to-one
  - `visits` to `a1c_long`: One-to-one (or zero-to-one for missing A1c values)

### 2.2 Stack the two visit files

In [24]:
visits_south_fixed = raw["visits_south"].rename(columns={"wt_kg": "weight_kg"})
north_copy = raw["visits_north"].copy(deep=True)
north_copy["source_file"] = "visits_north.csv"
south_copy = visits_south_fixed.copy(deep=True)
south_copy["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north_copy, south_copy], ignore_index=True)
print("Visits raw shape:", visits_raw.shape)
visits_raw

Visits raw shape: (57371, 7)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57366,P24976,week12,2026-05-14,10:50,153,78.3,visits_south.csv
57367,P19276,baseline,2026-01-22,15:35,160,122.2,visits_south.csv
57368,P21334,week12,2026-05-05,09:40,129,98.2,visits_south.csv
57369,P03524,baseline,2026-02-06,10:05,160,117.6,visits_south.csv


### 2.3 Audit the raw tables

In [25]:
p_raw = raw["patients"]
s_raw = raw["sites"]

# 1. gender not a code
valid_genders = {"F", "M", "X"}
g_stripped = p_raw["gender"].astype("string").str.strip()
non_code_gender = (~g_stripped.isin(valid_genders)) & (g_stripped != "") & (g_stripped.notna())

# 2. empty gender
empty_gender = (g_stripped == "") | g_stripped.isna()

# 3. birth_year 9999
by_raw = pd.to_numeric(p_raw["birth_year"], errors="coerce")
sentinel_by = (by_raw == 9999)

# 4. site codes needing normalization (changes when upper, spaces/hyphens removed)
sc_norm = p_raw["site_code"].astype("string").str.upper().str.replace("-", "").str.replace(" ", "")
sc_orig = p_raw["site_code"].astype("string")
site_needing_norm = (sc_norm != sc_orig)

# 5. repeated site_code in sites.csv
repeated_sites = s_raw.duplicated(subset=["site_code"], keep=False).sum()

# 6. visit labels not baseline or week12
v_labels = visits_raw["visit"].astype("string").str.strip().str.lower()
valid_visits = {"baseline", "week12"}
bad_visit_labels = (~v_labels.isin(valid_visits))

# 7. sbp sentinel 999
sbp_num = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")
sbp_sentinel = (sbp_num == 999)

# 8. sbp outside 70 through 250 (excluding 999)
sbp_outside = (sbp_num.notna()) & (sbp_num != 999) & ((sbp_num < 70) | (sbp_num > 250))

# 9. date parse failures (exact YYYY-MM-DD calendar date)
parsed_dates = pd.to_datetime(visits_raw["visit_date"], format="%Y-%m-%d", errors="coerce")
date_failures = parsed_dates.isna() | (visits_raw["visit_date"].astype("string").str.strip() != parsed_dates.dt.strftime("%Y-%m-%d"))

# 10. repeated patient_id and visit pairs (mapped to baseline/week12 first)
mapped_visit = v_labels.map({"baseline": "baseline", "wk12": "week12", "week12": "week12", "base": "baseline"}) # basic mapping representation for audit
repeat_pairs = visits_raw.assign(_v=mapped_visit).duplicated(subset=["patient_id", "_v"], keep=False).sum()

audit_data = [
    ["patients: gender values that are not a code", int(non_code_gender.sum())],
    ["patients: empty gender", int(empty_gender.sum())],
    ["patients: birth_year sentinel 9999", int(sentinel_by.sum())],
    ["patients: site codes needing normalization", int(site_needing_norm.sum())],
    ["sites: rows with a repeated site_code", int(repeated_sites)],
    ["visits: visit labels that are not baseline or week12", int(bad_visit_labels.sum())],
    ["visits: sbp sentinel 999", int(sbp_sentinel.sum())],
    ["visits: sbp outside 70 through 250", int(sbp_outside.sum())],
    ["visits: date parse failures", int(date_failures.sum())],
    ["visits: rows in repeated patient_id and visit pairs", int(repeat_pairs)]
]

issue_audit_df = pd.DataFrame(audit_data, columns=["issue", "count"])
ISSUE_AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
issue_audit_df.to_csv(ISSUE_AUDIT_PATH, index=False)
issue_audit_df

,issue,count
0,patients: gender values that are not a code,6540
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,1199
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,330


## Task 3: Clean

### 3.1 Clean the patients

In [26]:
p_df = raw["patients"].copy(deep=True)
p_df = p_df.drop_duplicates().copy()

g_map = {"female": "F", "f": "F", "male": "M", "m": "M", "nonbinary": "X", "x": "X"}
cleaned_gender = p_df["gender"].astype("string").str.strip().str.lower().map(g_map)
p_df["gender"] = cleaned_gender.astype("category")

by_num = pd.to_numeric(p_df["birth_year"], errors="coerce")
p_df["birth_year"] = by_num.mask(by_num == 9999).astype("Int64")

p_df["site_code"] = p_df["site_code"].astype("string").str.upper().str.replace("-", "").str.replace(" ", "")

patients_clean = p_df.reset_index(drop=True)



In [27]:
# Save patients summary
patients_summary = pd.DataFrame({
    "metric": [
        "patients", "birth_year missing", "birth_year min", "birth_year max",
        "site_code values", "patients at N01", "patients at S02", "patients at E04"
    ],
    "value": [
        len(patients_clean),
        int(patients_clean["birth_year"].isna().sum()),
        int(patients_clean["birth_year"].min()),
        int(patients_clean["birth_year"].max()),
        int(patients_clean["site_code"].nunique()),
        int((patients_clean["site_code"] == "N01").sum()),
        int((patients_clean["site_code"] == "S02").sum()),
        int((patients_clean["site_code"] == "E04").sum())
    ]
})
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

# Save gender categories
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts.columns = ["gender", "count"]
gender_counts["code"] = gender_counts["gender"].cat.codes
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

patients_summary

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


### 3.2 Clean the visits

In [28]:
v_df = visits_raw.copy(deep=True)
v_df = v_df.drop_duplicates().copy()

v_map = {"baseline": "baseline", "base": "baseline", "week12": "week12", "wk12": "week12"}
v_df["visit"] = v_df["visit"].astype("string").str.strip().str.lower().map(v_map)

sbp_num = pd.to_numeric(v_df["sbp_mmhg"], errors="coerce")
v_df["sbp_mmhg"] = sbp_num.mask((sbp_num == 999) | (sbp_num < 70) | (sbp_num > 250)).astype("Int64")

v_df["weight_kg"] = pd.to_numeric(v_df["weight_kg"], errors="coerce")

dt_parsed = pd.to_datetime(v_df["visit_date"], format="%Y-%m-%d", errors="coerce")
valid_date_mask = dt_parsed.notna() & (v_df["visit_date"].astype("string").str.strip() == dt_parsed.dt.strftime("%Y-%m-%d"))
v_df["visit_date"] = dt_parsed.mask(~valid_date_mask)

In [29]:
repeated_pairs = v_df[v_df.duplicated(subset=["patient_id", "visit"], keep=False)]
print(f"Found {len(repeated_pairs)} rows in repeated patient_id and visit pairs (rechecks):")
display(repeated_pairs.head())

v_df = v_df.sort_values("visit_time")
v_df = v_df.drop_duplicates(subset=["patient_id", "visit"], keep="last")

Found 190 rows in repeated patient_id and visit pairs (rechecks):


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152,68.2,visits_north.csv
1208,P01341,week12,2026-05-26,15:10,121,72.3,visits_north.csv


In [30]:
v_df = v_df.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = v_df[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]].copy()

# Save visits summary
visits_summary = pd.DataFrame({
    "metric": [
        "visits", "baseline visits", "week12 visits", "visit_date missing",
        "visit_date min", "visit_date max", "sbp_mmhg missing", "sbp_mmhg min",
        "sbp_mmhg max", "weight_kg missing", "weight_kg mean",
        "visits from visits_north.csv", "visits from visits_south.csv"
    ],
    "value": [
        len(visits_clean),
        int((visits_clean["visit"] == "baseline").sum()),
        int((visits_clean["visit"] == "week12").sum()),
        int(visits_clean["visit_date"].isna().sum()),
        str(visits_clean["visit_date"].min()),
        str(visits_clean["visit_date"].max()),
        int(visits_clean["sbp_mmhg"].isna().sum()),
        int(visits_clean["sbp_mmhg"].min()) if visits_clean["sbp_mmhg"].notna().any() else None,
        int(visits_clean["sbp_mmhg"].max()) if visits_clean["sbp_mmhg"].notna().any() else None,
        int(visits_clean["weight_kg"].isna().sum()),
        round(float(visits_clean["weight_kg"].mean()), 2) if visits_clean["weight_kg"].notna().any() else None,
        int((visits_clean["source_file"] == "visits_north.csv").sum()),
        int((visits_clean["source_file"] == "visits_south.csv").sum())
    ]
})
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary

,metric,value
0,visits,57206
1,baseline visits,29164
2,week12 visits,27229
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [31]:
decisions = [
    ["patients", "exact duplicate rows", "Dropped exact duplicates, keeping the first occurrence.", "Eliminates redundant entries to maintain a unique patient grain."],
    ["patients", "gender spellings that are not a code", "Mapped variations like 'female' or 'woman' to standard codes and unknown entries to missing.", "Standardizes categorical values into official F, M, X codes."],
    ["patients", "site codes needing normalization", "Uppercased and stripped all spaces and hyphens.", "Ensures uniform site identifiers across files."],
    ["patients", "birth_year sentinel 9999", "Converted 9999 to missing (NaN) with Int64 type.", "Removes unrecorded sentinel values so they do not skew numeric summaries."],
    ["visits", "weight column named wt_kg in visits_south.csv", "Renamed wt_kg to weight_kg prior to stacking.", "Harmonizes column schemas between north and south clinic export files."],
    ["visits", "sbp sentinel 999 and readings outside 70 through 250", "Coerced invalid measurements and 999 sentinels to missing.", "Prevents erroneous or placeholder blood pressure readings from distorting analyses."],
    ["visits", "impossible and loosely written visit dates", "Coerced non-calendar or improperly formatted date text to missing.", "Enforces strict calendar date validation."],
    ["visits", "repeated patient_id and visit pairs", "Kept the later reading ordered by visit_time for duplicate patient-visit pairs.", "Resolves clinic rechecks by retaining the final recorded measurement."],
    ["sites", "repeated site_code rows", "Filtered to keep only current active site records.", "Removes retired records to maintain a clean site directory."]
]

decision_table = pd.DataFrame(decisions, columns=["table", "issue", "decision", "reason"])
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,"Dropped exact duplicates, keeping the first oc...",Eliminates redundant entries to maintain a uni...
1,patients,gender spellings that are not a code,Mapped variations like 'female' or 'woman' to ...,Standardizes categorical values into official ...
2,patients,site codes needing normalization,Uppercased and stripped all spaces and hyphens.,Ensures uniform site identifiers across files.
3,patients,birth_year sentinel 9999,Converted 9999 to missing (NaN) with Int64 type.,Removes unrecorded sentinel values so they do ...
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg prior to stacking.,Harmonizes column schemas between north and so...
5,visits,sbp sentinel 999 and readings outside 70 throu...,Coerced invalid measurements and 999 sentinels...,Prevents erroneous or placeholder blood pressu...
6,visits,impossible and loosely written visit dates,Coerced non-calendar or improperly formatted d...,Enforces strict calendar date validation.
7,visits,repeated patient_id and visit pairs,Kept the later reading ordered by visit_time f...,Resolves clinic rechecks by retaining the fina...
8,sites,repeated site_code rows,Filtered to keep only current active site reco...,Removes retired records to maintain a clean si...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [32]:
a1c_w = raw["a1c_wide"]
a1c_long = a1c_w.melt(id_vars=["patient_id"], value_vars=["a1c_baseline", "a1c_week12"], var_name="visit", value_name="a1c_pct")
a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})
a1c_long["a1c_pct"] = pd.to_numeric(a1c_long["a1c_pct"], errors="coerce")
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [
        int((a1c_long["visit"] == "baseline").sum()),
        int((a1c_long["visit"] == "week12").sum())
    ],
    "a1c_pct_mean": [
        round(float(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean()), 2),
        round(float(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean()), 2)
    ]
})
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)
a1c_summary

,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [33]:
sites = raw["sites"]
try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("Caught expected MergeError:", e)

repeated_site_rows = sites[sites.duplicated(subset=["site_code"], keep=False)]
print("\nRepeated site rows:")
display(repeated_site_rows)

sites_current = sites[sites["record_status"].str.strip().str.lower() == "current"][["site_code", "site_name"]].copy()

Caught expected MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...

Repeated site rows:


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


In [34]:
merge_check = visits_clean.merge(a1c_long, on=["patient_id", "visit"], how="outer", indicator=True)
print("\nMerge indicator value counts:")
print(merge_check["_merge"].value_counts())

# Save counts to output/merge_audit.csv
visits_with_a1c = int((merge_check["_merge"] == "both").sum())
visits_without_a1c = int((merge_check["_merge"] == "left_only").sum())
a1c_without_visit = int((merge_check["_merge"] == "right_only").sum())
visits_not_in_registry = int((~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum())
patients_no_visit = int((~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum())
patients_bad_site = int((~patients_clean["site_code"].isin(sites["site_code"])).sum())
sites_removed = int(len(sites) - len(sites_current))

merge_audit_df = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule"
    ],
    "count": [
        visits_with_a1c,
        visits_without_a1c,
        a1c_without_visit,
        visits_not_in_registry,
        patients_no_visit,
        patients_bad_site,
        sites_removed
    ]
})

MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
merge_audit_df.to_csv(MERGE_AUDIT_PATH, index=False)
merge_audit_df


Merge indicator value counts:
_merge
both          53257
left_only      3949
right_only     3503
Name: count, dtype: int64


,check,count
0,visits with an A1c result,53257
1,visits without an A1c result,3949
2,A1c results without a visit,3503
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [35]:
sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None
if "baseline" in sbp_wide.columns and "week12" in sbp_wide.columns:
    sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]
else:
    sbp_wide["sbp_change"] = pd.NA

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)
sbp_wide_sample

,patient_id,NaN,baseline,week12,sbp_change
0,P00001,<NA>,146,129,-17
1,P00004,<NA>,142,<NA>,<NA>
2,P00088,<NA>,152,147,-5
3,P00260,<NA>,70,138,68
4,P00446,<NA>,250,144,-106
5,P00487,<NA>,<NA>,143,<NA>
6,P00851,<NA>,<NA>,129,<NA>
7,P02354,<NA>,127,125,-2
8,P03836,<NA>,134,125,-9
9,P05763,<NA>,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [36]:
# Select complete pairs (patients with both baseline and week12 readings), in that order
complete_pairs = sbp_wide.dropna(subset=["baseline", "week12"])[["baseline", "week12"]]

# Convert to a NumPy array with float dtype
arr = np.array(complete_pairs, dtype=float)

# Display array shape and calculate per-visit means using NumPy
print("Array shape:", arr.shape)
means = arr.mean(axis=0)
print("Per-visit means (baseline, week12) in mmHg:", means)

Array shape: (26409, 2)
Per-visit means (baseline, week12) in mmHg: [143.02415843 137.16183877]


### 4.4 Join the analysis table

In [43]:
analysis = visits_clean.merge(patients_clean, on="patient_id", how="left", validate="many_to_one")
analysis = analysis.merge(sites_current, on="site_code", how="left", validate="many_to_one")
analysis = analysis.merge(a1c_long, on=["patient_id", "visit"], how="left", validate="many_to_many")

analysis = analysis[[
    "patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "a1c_pct",
    "gender", "birth_year", "site_code", "site_name", "source_file"
]]

analysis["visit_date"] = pd.to_datetime(analysis["visit_date"]).astype("datetime64[us]")

analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string"
})
print("Analysis shape:", analysis.shape)
print(analysis.dtypes)

Analysis shape: (57206, 11)
patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object


## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [38]:
checks = pd.Series({
    "analysis row count matches visits_clean": len(analysis) == len(visits_clean),
    "no patient_id and visit pair repeats in analysis": not analysis.duplicated(subset=["patient_id", "visit"]).any(),
    "sbp values within 70-250 range": ((analysis["sbp_mmhg"] >= 70) & (analysis["sbp_mmhg"] <= 250)).all() or analysis["sbp_mmhg"].isna().all(),
    "site_name is current": analysis["site_name"].dropna().isin(sites_current["site_name"]).all(),
    "sbp_wide patient_id is unique": sbp_wide["patient_id"].is_unique,
})

print(checks)
assert checks.all(), checks[~checks]
print("All validation checks passed successfully!")

analysis row count matches visits_clean             True
no patient_id and visit pair repeats in analysis    True
sbp values within 70-250 range                      True
site_name is current                                True
sbp_wide patient_id is unique                       True
dtype: bool
All validation checks passed successfully!


### 5.2 Save the analysis extract

In [39]:
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)
analysis_sample

,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,<NA>,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,<NA>,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,<NA>,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,week12,2026-05-26,143,NaN,7.8,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [40]:
round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string"
    },
    parse_dates=["visit_date"]
)

is_equal = round_trip.equals(analysis_sample)
print("Round trip equals original:", is_equal)

Round trip equals original: True
